# Lab 3A — Build a Real Vector Search Index

[Lab 2B](#) ended badly on purpose: the keyword retriever scored the correct
document at **0.0000**, and you could not tell from the agent's answer.

This lab replaces it with a managed **Mosaic AI Vector Search** index — and then
tests it with questions whose right answer you already know, because "it feels
better" is not a measurement.

| Cell group | What you do |
|---|---|
| 0–2 | Chunk the documents, carrying metadata with each chunk. |
| 3 | Turn on the **two table properties** Vector Search requires. |
| 4–5 | Create the endpoint and the Delta Sync index, and wait for it. |
| 6 | **Four known-answer retrieval checks.** |
| 7 | The `audience` filter as a governance boundary. |

**Compute:** Serverless.

In [0]:
%pip install -q databricks-ai-search
dbutils.library.restartPython()

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


## 1. Why chunk at all?

In [Lab 2A](#) the retriever returned whole documents. That is fine for seven short
policies and hopeless at any real size: you hand the model 2,000 words to answer a
question a single sentence covers, and the relevant sentence competes with everything
else for attention.

**Chunking is a retrieval decision, not a storage one.** The unit you index is the
unit that gets retrieved.

In [0]:
CATALOG, SCHEMA = "agents_labs", "retail"
SRC   = f"{CATALOG}.{SCHEMA}.support_docs"
CHUNKS = f"{CATALOG}.{SCHEMA}.support_chunks"
INDEX = f"{CATALOG}.{SCHEMA}.support_chunks_idx"
ENDPOINT = "agents-labs-vs"

docs = [r.asDict() for r in spark.sql(f"SELECT * FROM {SRC} ORDER BY doc_id").collect()]
print(f"  {len(docs)} source documents")
for d in docs:
    print(f"    {d['doc_id']}  {d['audience']:11} {len(d['body'].split()):4d} words  {d['title']}")

  7 source documents
    DOC-001  customer     127 words  Returns and refunds policy
    DOC-002  agent_only   114 words  Lost in transit procedure
    DOC-003  customer      85 words  Delivery timescales by region
    DOC-004  customer      95 words  Warranty coverage
    DOC-005  customer      88 words  Billing, invoices and payment terms
    DOC-006  agent_only    89 words  Refund authority limits
    DOC-007  customer      75 words  Bulk and fit-out orders


## 2. Chunk — and carry the metadata with you

Sentence-aware, three sentences per chunk. Crude, and good enough for policy prose
that is already written in short declarative sentences.

**The part that matters is not the splitting.** It is that every chunk keeps its
`doc_id`, `title`, `category` and — critically — its **`audience`**.

A chunk that loses its `audience` is a chunk you cannot filter. You would be relying
on the model not to quote it.

In [0]:
import re

def chunk(body, size=3):
    sents = [s.strip() for s in re.split(r"(?<=[.!?])\s+", body.strip()) if s.strip()]
    return [" ".join(sents[i:i+size]) for i in range(0, len(sents), size)]

rows = []
for d in docs:
    for i, text in enumerate(chunk(d["body"])):
        rows.append((f"{d['doc_id']}-C{i:02d}", d["doc_id"], d["title"],
                     d["category"], d["audience"], text))

print(f"  {len(docs)} documents -> {len(rows)} chunks\n")
for cid, did, title, cat, aud, text in rows[:4]:
    print(f"  {cid}  {aud:11} {text[:66]}...")

  7 documents -> 15 chunks

  DOC-001-C00  customer    Customers may return any item within 30 days of delivery for a ful...
  DOC-001-C01  customer    Return shipping is free for premier tier customers; standard and p...
  DOC-002-C00  agent_only  An order may be marked lost_in_transit only after the courier has ...
  DOC-002-C01  agent_only  For orders of 50 GBP or more, a supervisor must approve the refund...


## 3. The two properties Vector Search requires

A Delta Sync index does not read your table once. It **follows** it. For that,
Databricks needs two things, and it will refuse the index without them:

| Requirement | Why |
|---|---|
| **Change Data Feed** enabled | so the index can see what changed, rather than re-reading everything |
| A **primary key** | so a changed row updates in place instead of duplicating |

This cell is safe to re-run. If the table already exists with rows, it leaves it
alone — rebuilding the source of a live index forces a full re-sync.

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

exists = spark.catalog.tableExists(CHUNKS)
n = spark.table(CHUNKS).count() if exists else 0

if n:
    print(f"  {CHUNKS} already exists with {n} rows — leaving it alone")
else:
    schema = StructType([StructField(c, StringType(), c != "chunk_id")
                         for c in ["chunk_id","doc_id","title","category","audience","chunk"]])
    spark.createDataFrame(rows, schema).write.mode("overwrite").saveAsTable(CHUNKS)
    spark.sql(f"ALTER TABLE {CHUNKS} SET TBLPROPERTIES (delta.enableChangeDataFeed = true)")
    spark.sql(f"ALTER TABLE {CHUNKS} ALTER COLUMN chunk_id SET NOT NULL")
    spark.sql(f"ALTER TABLE {CHUNKS} ADD CONSTRAINT support_chunks_pk PRIMARY KEY (chunk_id)")
    print(f"  created {CHUNKS} with {len(rows)} rows, CDF on, PK on chunk_id")

props = {r["key"]: r["value"] for r in spark.sql(f"SHOW TBLPROPERTIES {CHUNKS}").collect()}
print(f"\n  delta.enableChangeDataFeed = {props.get('delta.enableChangeDataFeed')}")
print(f"  rows                       = {spark.table(CHUNKS).count()}")

  agents_labs.retail.support_chunks already exists with 15 rows — leaving it alone

  delta.enableChangeDataFeed = true
  rows                       = 15


## 4. The endpoint

An **endpoint** is the compute that serves indexes; an **index** lives on one. They
are billed and managed separately, and the endpoint is the expensive one.

> ⚠️ In a class, the endpoint is usually **shared and already created**. This cell
> only creates one if it is missing.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.ai_search.client import VectorSearchClient

w = WorkspaceClient()
vsc = VectorSearchClient(
    workspace_url=w.config.host,
    personal_access_token=w.config.authenticate()["Authorization"].removeprefix("Bearer "),
    disable_notice=True,
)

names = [e["name"] for e in vsc.list_endpoints().get("endpoints", [])]
if ENDPOINT in names:
    print(f"  endpoint '{ENDPOINT}' already exists")
else:
    vsc.create_endpoint(name=ENDPOINT, endpoint_type="STANDARD")
    print(f"  creating endpoint '{ENDPOINT}' — this takes several minutes")

ep = vsc.get_endpoint(ENDPOINT)
print(f"  state: {ep.get('endpoint_status', {}).get('state')}")

  endpoint 'agents-labs-vs' already exists
  state: ONLINE


## 5. The index — managed embeddings

`DELTA_SYNC` with `embedding_source_column` means **Databricks embeds for you**.
You never call an embeddings model, never store a vector, and never write the code
that keeps them in step with the table.

Compare that with [Lab 2B](#)'s `EmbeddingRetriever`, which re-embedded all seven
documents on every notebook run and would have silently gone stale the moment a
document changed.

`pipeline_type="TRIGGERED"` syncs when asked. `CONTINUOUS` keeps a stream running —
lower latency, always-on cost.

In [0]:
import time

existing = [i["name"] for i in vsc.list_indexes(name=ENDPOINT).get("vector_indexes", [])]

if INDEX in existing:
    print(f"  index '{INDEX}' already exists")
else:
    vsc.create_delta_sync_index(
        endpoint_name=ENDPOINT,
        index_name=INDEX,
        source_table_name=CHUNKS,
        pipeline_type="TRIGGERED",
        primary_key="chunk_id",
        embedding_source_column="chunk",
        embedding_model_endpoint_name="databricks-gte-large-en",
    )
    print(f"  creating index '{INDEX}' ...")

ix = vsc.get_index(endpoint_name=ENDPOINT, index_name=INDEX)
for _ in range(60):
    d = ix.describe()
    st = d.get("status", {})
    if st.get("ready"):
        print(f"  READY   indexed rows: {st.get('indexed_row_count')}")
        break
    print(f"  {st.get('detailed_state')} ...")
    time.sleep(20)

  index 'agents_labs.retail.support_chunks_idx' already exists
  READY   indexed rows: 15


## 6. Prove it retrieves — with known answers

This is the cell that matters.

Four questions. For each, **you already know which document should come back first**.
That is what makes this a test rather than a demo. Case 2 is the paraphrase that
scored `0.0000` under keyword retrieval in [Lab 2B](#).

In [0]:
CASES = [
    ("How long does delivery to the Nordics take?",   "customer",   "DOC-003"),
    ("Can I send it back after six weeks?",           "customer",   "DOC-001"),
    ("How much can I refund without asking anyone?",  "agent_only", "DOC-006"),
    ("What is covered if the chair frame breaks?",    "customer",   "DOC-004"),
]

passed = 0
for q, aud, expect in CASES:
    r = ix.similarity_search(query_text=q,
                             columns=["chunk_id", "doc_id", "title", "audience"],
                             filters={"audience": aud}, num_results=3)
    rows_ = r.get("result", {}).get("data_array", []) or []
    got = [x[1] for x in rows_]
    ok = bool(got) and got[0] == expect
    passed += ok
    print(f"\n  q: {q}")
    print(f"     filter=audience:{aud}   expect top-1={expect}   {'PASS' if ok else 'FAIL'}")
    for x in rows_:
        print(f"       {x[0]:12} {x[1]}  {x[3]:10} score={x[-1]:.4f}  {x[2][:40]}")

print(f"\n  {passed}/{len(CASES)} retrieval checks passed")

[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

  q: How long does delivery to the Nordics take?
     filter=audience:customer   expect top-1=DOC-003   PASS
       DOC-003-C00  DOC-003  customer   score=0.7114  Delivery timescales by region
       DOC-003-C01  DOC-003  customer   score=0.5564  Delivery timescales by region
       DOC-005-C00  DOC-005  customer   score=0.5522  Billing, invoices and payment terms
[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

  q: Can I send it back after six weeks?
     filter=audience:customer   expect top-1=DOC-001   PASS
       DOC-001-C00  DOC-001  customer   score=0.5520  Returns and refunds policy
       DOC-007-C0

## 7. The filter is a governance boundary

Case 3 asked *"How much can I refund without asking anyone?"* with
`audience: agent_only` and got `DOC-006`.

Ask the **same question** as a customer and watch what happens.

In [0]:
q = "How much can I refund without asking anyone?"

for aud in ("agent_only", "customer"):
    r = ix.similarity_search(query_text=q,
                             columns=["chunk_id", "doc_id", "title", "audience"],
                             filters={"audience": aud}, num_results=3)
    print(f"\n  filters={{'audience': '{aud}'}}")
    for x in (r.get("result", {}).get("data_array", []) or []):
        print(f"      {x[1]}  {x[3]:10} score={x[-1]:.4f}  {x[2][:44]}")

print("\n  DOC-006 is unreachable as a customer. The question is still asked;")
print("  the answer simply is not in the candidate set.")

[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

  filters={'audience': 'agent_only'}
      DOC-006  agent_only score=0.5974  Refund authority limits
      DOC-006  agent_only score=0.5647  Refund authority limits
      DOC-002  agent_only score=0.5186  Lost in transit procedure
[NOTICE] Using a Personal Authentication Token (PAT). Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.

  filters={'audience': 'customer'}
      DOC-001  customer   score=0.5161  Returns and refunds policy
      DOC-001  customer   score=0.5112  Returns and refunds policy
      DOC-007  customer   score=0.4879  Bulk and fit-out orders

  DOC-006 is unreachable as a customer. The question is still asked;
  the answer simply is not in th

## 8. Where that boundary actually lives

The filter worked. Now be precise about **what protected you**.

| | |
|---|---|
| What stopped the disclosure | the `filters=` argument, in the calling code |
| What did **not** stop it | any grant, policy or property in Unity Catalog |

The index contains `DOC-006`. Every caller with access to the index can read it.
The only thing keeping it from customers is that **this** caller remembered to pass
`filters={"audience": "customer"}`.

> 🚨 **That is a convention, not a control.** In [Lab 7B](#) a no-code Agent Bricks
> assistant is pointed at this same index. It has no filter control at all — and it
> quotes `DOC-006` verbatim to a customer-facing chatbot.
>
> The fix is not a better prompt. It is to put internal content where the
> customer-facing agent has no grant: a separate index, or an index built over a
> view filtered to `audience = 'customer'`.

### Try this before you move on

1. Drop `filters=` entirely from cell 7 and re-run. What comes back for the refund
   question now?
2. Add a row to `support_chunks`, then re-run cell 6 **without** re-syncing. Is it
   found? Now call `ix.sync()` and try again — that is what `TRIGGERED` means.
3. Compare the top score here with the `0.0000` from Lab 2B for the same paraphrase.